# Modelling Pipeline

In [1]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
train_df = pd.read_csv("census_income_learn.csv", header=None)
test_df  = pd.read_csv("census_income_test.csv", header=None)

In [3]:
columns = [
    "age","class_of_worker","industry_code","occupation_code","education",
    "wage_per_hour","enrolled_in_edu","marital_status","major_industry_code",
    "major_occupation_code","race","hispanic_origin","sex",
    "member_of_labor_union","reason_for_unemployment","employment_stat",
    "capital_gains","capital_losses","dividends","tax_filer_status",
    "region","state","household_status","household_summary",
    "instance_weight","migration_code","migration_reg","migration_msa",
    "migration_within_state","migration_sunbelt","num_persons_employer",
    "family_members_under_18","country_of_birth_father",
    "country_of_birth_mother","country_of_birth_self","citizenship",
    "own_business","fill_inc_questionnaire","veterans_benefits",
    "weeks_worked","year","income"
]

train_df.columns = columns
test_df.columns  = columns

In [4]:
def clean_census(df):
    df = df.copy()
    
    # Strip whitespace
    obj_cols = df.select_dtypes(include="object").columns
    df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())
    
    # Normalise unknowns
    df[obj_cols] = df[obj_cols].replace({"?": "UNKNOWN"})
    
    # Normalise NOT IN UNIVERSE variants
    df[obj_cols] = df[obj_cols].replace({
        "Not in universe": "NOT_IN_UNIVERSE",
        "Not in universe or children": "NOT_IN_UNIVERSE",
        "Not in universe under 1 year old": "NOT_IN_UNIVERSE"
    })
    
    # Target
    df["income_over_50k"] = (df["income"] == "50000+.").astype(int)
    
    # Drop raw income string
    df = df.drop(columns=["income"])
    
    return df

In [5]:
train_df = clean_census(train_df)
test_df  = clean_census(test_df)

In [6]:
train_df["income_over_50k"].value_counts(normalize=True)

income_over_50k
0    0.937942
1    0.062058
Name: proportion, dtype: float64

In [7]:
test_df["income_over_50k"].value_counts(normalize=True)

income_over_50k
0    0.937992
1    0.062008
Name: proportion, dtype: float64

In [8]:
TARGET = "income_over_50k"
WEIGHT = "instance_weight"

In [9]:
DROP_COLS = ["fill_inc_questionnaire"]

train_df = train_df.drop(columns=DROP_COLS)
test_df  = test_df.drop(columns=DROP_COLS)

# define feature lists
TARGET = "income_over_50k"
WEIGHT = "instance_weight"

num_cols = [
    "age",
    "wage_per_hour",
    "capital_gains",
    "capital_losses",
    "dividends",
    "weeks_worked"
]

coded_num_cols = [
    "industry_code",
    "occupation_code",
    "own_business",
    "veterans_benefits",
    "num_persons_employer",
    "year"
]

cat_cols = [
    c for c in train_df.columns
    if c not in num_cols + coded_num_cols + [TARGET, WEIGHT]
    and train_df[c].dtype == "object"
]

In [10]:
train_df[coded_num_cols] = train_df[coded_num_cols].astype("category")
test_df[coded_num_cols]  = test_df[coded_num_cols].astype("category")

In [22]:
train_df[WEIGHT]

0         1700.09
1         1053.55
2          991.95
3         1758.14
4         1069.16
           ...   
199518     955.27
199519     687.19
199520    1923.03
199521    4664.87
199522    1830.11
Name: instance_weight, Length: 199523, dtype: float64

In [11]:
for col in ["capital_gains", "capital_losses", "dividends"]:
    train_df[col] = np.log1p(train_df[col])
    test_df[col]  = np.log1p(test_df[col])


In [12]:
X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
w_train = train_df[WEIGHT]

X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]
w_test  = test_df[WEIGHT]

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=100
        ), cat_cols + coded_num_cols)
    ]
)

In [14]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=500,
        class_weight="balanced"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=50,
        n_jobs=-1,
        random_state=42
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )
}
# Xgboost, CatBoost

In [ ]:
# Regression models
# LinearRegressor, RidgeRegressor, LassoRegressor, ElasticNetRegressor,
# RandomForestRegressor, XgboostRegressor

In [15]:
results = []

for name, model in models.items():
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train, model__sample_weight=w_train)
    
    y_pred = pipe.predict(X_test)
    
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred, sample_weight=w_test),
        "precision": precision_score(y_test, y_pred, sample_weight=w_test),
        "recall": recall_score(y_test, y_pred, sample_weight=w_test),
        "f1": f1_score(y_test, y_pred, sample_weight=w_test)
    })

C:\Users\tjani\Anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [16]:
results_df = pd.DataFrame(results).sort_values("f1", ascending=False)
results_df

,model,accuracy,precision,recall,f1
2,GradientBoosting,0.954492,0.756808,0.415750,0.536678
0,LogisticRegression,0.852333,0.285903,0.887591,0.432495
1,RandomForest,0.943245,0.910869,0.116104,0.205956


In [17]:
best_model_name = results_df.iloc[0]["model"]
best_model_name

'GradientBoosting'

In [18]:
best_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", models[best_model_name])
])

best_pipeline.fit(X_train, y_train, model__sample_weight=w_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'wage_per_hour',
                                                   'capital_gains',
                                                   'capital_losses',
                                                   'dividends',
                                                   'weeks_worked']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=100),
                                                  ['class_of_worker',
                                                   'education',
                                                   'enrolled_in_edu',
                                                   'marital_status',
                                                   'major_industry_code',
                                                   'major_occupation_code',
                                                   'race', 'h...
                                                   'migration_code',
                                                   'migration_reg',
                                                   'migration_msa',
                                                   'migration_within_state',
                                                   'migration_sunbelt',
                                                   'family_members_under_18',
                                                   'country_of_birth_father',
                                                   'country_of_birth_mother',
                                                   'country_of_birth_self',
                                                   'citizenship',
                                                   'industry_code',
                                                   'occupation_code',
                                                   'own_business', ...])])),
                ('model',
                 GradientBoostingClassifier(learning_rate=0.05,
                                            n_estimators=200,
                                            random_state=42))])

In [19]:
y_final = best_pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_final, sample_weight=w_test))
print("Precision:", precision_score(y_test, y_final, sample_weight=w_test))
print("Recall   :", recall_score(y_test, y_final, sample_weight=w_test))
print("F1       :", f1_score(y_test, y_final, sample_weight=w_test))

Accuracy : 0.9544923687670588
Precision: 0.7568077963608765
Recall   : 0.41575039740448294
F1       : 0.5366780834739681


In [20]:
import joblib
joblib.dump(best_pipeline, "income_model.joblib")

['income_model.joblib']

## Results summary

- Report best model + metrics
- Summarise key drivers (LogReg coefficients or permutation importance)
- Provide recommendations for production hardening and monitoring
